# Eksperimen Revisi JTIE #646
Menjawab permintaan reviewer:
- **R3**: ulangi dengan beberapa *seed*, augmentasi untuk **ketiga** model, uji statistik berpasangan + interval kepercayaan
- **R1**: benchmark **CPU**, prosedur waktu yang jelas (sinkronisasi CUDA, dengan/tanpa preprocessing, rata-rata ± SD), **memori puncak** saat inferensi
- **R2**: analisis **verifikasi manusia berbasis confidence**, file *split* & skrip untuk repositori

Cara pakai: **Runtime → Change runtime type → T4 GPU**, lalu **Runtime → Run all**. Saat diminta, **izinkan akses Google Drive**. Hasil tiap run tersimpan di folder Drive `hasil_revisi_jtie`, jadi kalau koneksi putus cukup **Run all** lagi: run yang sudah selesai otomatis dilewati.
Hasil akhir terunduh otomatis sebagai `hasil_revisi_jtie.zip`.

## 1. Pengaturan

In [ ]:
SEEDS            = [42, 123, 2026]   # 3 seed pelatihan
SPLIT_SEED       = 42                # split data tetap sama dengan naskah
JUMLAH_KELAS     = 10
MAKS_CITRA_KELAS = 400
EPOCH            = 15
BATCH_SIZE       = 32
LEARNING_RATE    = 1e-3
UKURAN_CITRA     = 224
MODEL            = ['MobileNetV3-Small', 'EfficientNet-B0', 'ShuffleNetV2']
SKENARIO_AUG     = [True, False]     # dengan & tanpa augmentasi untuk SEMUA model
N_BOOT           = 2000              # bootstrap untuk interval kepercayaan 95%

## 2. Pustaka

In [ ]:
!pip -q install kagglehub
import os, time, random, json, shutil, subprocess, sys, platform
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, f1_score
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.multitest import multipletests
import kagglehub
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
from google.colab import drive
drive.mount('/content/drive')   # hasil disimpan di Google Drive agar aman jika koneksi putus
OUT = '/content/drive/MyDrive/hasil_revisi_jtie'; os.makedirs(f'{OUT}/runs', exist_ok=True); os.makedirs(f'{OUT}/kode', exist_ok=True)
print('Perangkat:', DEVICE, torch.cuda.get_device_name(0) if DEVICE=='cuda' else '')
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

## 3. Dataset & split (identik dengan naskah, disimpan untuk repositori)

In [ ]:
path = kagglehub.dataset_download('paramaggarwal/fashion-product-images-small')
csv_path = img_dir = None
for root, dirs, files in os.walk(path):
    if 'styles.csv' in files and csv_path is None: csv_path = os.path.join(root, 'styles.csv')
    if os.path.basename(root) == 'images' and img_dir is None: img_dir = root
df = pd.read_csv(csv_path, on_bad_lines='skip')
df['path'] = df['id'].astype(str).apply(lambda i: os.path.join(img_dir, i + '.jpg'))
df = df[df['path'].apply(os.path.exists)]
top = df['articleType'].value_counts().head(JUMLAH_KELAS).index.tolist()
df = df[df['articleType'].isin(top)]
df = df.groupby('articleType', group_keys=False).apply(lambda g: g.sample(min(len(g), MAKS_CITRA_KELAS), random_state=SPLIT_SEED))
KELAS = sorted(top); idx = {k:i for i,k in enumerate(KELAS)}
df['label'] = df['articleType'].map(idx)
train_df, tmp = train_test_split(df, test_size=0.30, stratify=df['label'], random_state=SPLIT_SEED)
val_df, test_df = train_test_split(tmp, test_size=0.50, stratify=tmp['label'], random_state=SPLIT_SEED)
split = pd.concat([train_df.assign(subset='train'), val_df.assign(subset='val'), test_df.assign(subset='test')])
split[['id','articleType','label','subset']].to_csv(f'{OUT}/split_dataset.csv', index=False)
# distribusi kelas pada dataset penuh (untuk justifikasi sampling, R3-2)
full = pd.read_csv(csv_path, on_bad_lines='skip')['articleType'].value_counts()
full.head(30).to_csv(f'{OUT}/distribusi_kelas_dataset_asli.csv')
print('Total:', len(df), '| kelas:', KELAS)
print('Jumlah articleType di dataset asli:', full.size, '| 10 teratas mencakup', round(full.head(10).sum()/full.sum()*100,1), '% citra')

## 4. Preprocessing, model, latih & evaluasi

In [ ]:
MEAN, STD = [0.485,0.456,0.406], [0.229,0.224,0.225]
tf_aug = transforms.Compose([transforms.Resize((UKURAN_CITRA,UKURAN_CITRA)), transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15), transforms.RandomResizedCrop(UKURAN_CITRA, scale=(0.85,1.0)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2), transforms.ToTensor(), transforms.Normalize(MEAN,STD)])
tf_plain = transforms.Compose([transforms.Resize((UKURAN_CITRA,UKURAN_CITRA)), transforms.ToTensor(), transforms.Normalize(MEAN,STD)])
class DS(Dataset):
    def __init__(s, d, tf): s.p, s.y, s.tf = d['path'].tolist(), d['label'].tolist(), tf
    def __len__(s): return len(s.p)
    def __getitem__(s, i): return s.tf(Image.open(s.p[i]).convert('RGB')), s.y[i]
def buat_model(nama, n):
    if nama=='MobileNetV3-Small':
        m=models.mobilenet_v3_small(weights='IMAGENET1K_V1'); m.classifier[3]=nn.Linear(m.classifier[3].in_features,n)
    elif nama=='EfficientNet-B0':
        m=models.efficientnet_b0(weights='IMAGENET1K_V1'); m.classifier[1]=nn.Linear(m.classifier[1].in_features,n)
    else:
        m=models.shufflenet_v2_x1_0(weights='IMAGENET1K_V1'); m.fc=nn.Linear(m.fc.in_features,n)
    return m
def latih(nama, augment, seed, sabar=4):
    set_seed(seed); g = torch.Generator(); g.manual_seed(seed)
    tr = DataLoader(DS(train_df, tf_aug if augment else tf_plain), BATCH_SIZE, shuffle=True, num_workers=2, generator=g)
    va = DataLoader(DS(val_df, tf_plain), BATCH_SIZE, num_workers=2)
    m = buat_model(nama, len(KELAS)).to(DEVICE)
    opt = torch.optim.Adam(m.parameters(), lr=LEARNING_RATE)
    sch = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=2)
    lossf = nn.CrossEntropyLoss(); best, wait, bobot, ep_best = 1e9, 0, None, 0
    for ep in range(EPOCH):
        m.train()
        for x,y in tr:
            x,y = x.to(DEVICE), y.to(DEVICE); opt.zero_grad(); l = lossf(m(x),y); l.backward(); opt.step()
        m.eval(); vl, n = 0, 0
        with torch.no_grad():
            for x,y in va:
                x,y = x.to(DEVICE), y.to(DEVICE); vl += lossf(m(x),y).item()*len(y); n += len(y)
        vl/=n; sch.step(vl)
        if vl < best: best, wait, ep_best, bobot = vl, 0, ep+1, {k:v.detach().clone() for k,v in m.state_dict().items()}
        else:
            wait += 1
            if wait >= sabar: break
    m.load_state_dict(bobot); return m, ep_best
def prediksi(m):
    te = DataLoader(DS(test_df, tf_plain), BATCH_SIZE, num_workers=2); m.eval(); P=[]; Y=[]
    with torch.no_grad():
        for x,y in te:
            P.append(torch.softmax(m(x.to(DEVICE)),1).cpu().numpy()); Y += y.tolist()
    return np.array(Y), np.concatenate(P)

## 5. Jalankan 3 model × 2 skenario augmentasi × 3 seed (18 run)
Kalau koneksi terputus, jalankan ulang sel ini: run yang sudah selesai akan dilewati.

In [ ]:
for nama in MODEL:
    for aug in SKENARIO_AUG:
        for seed in SEEDS:
            tag = f"{nama}_{'aug' if aug else 'noaug'}_s{seed}"
            if os.path.exists(f'{OUT}/runs/{tag}.npz'): print('lewati', tag); continue
            t0 = time.time(); m, ep = latih(nama, aug, seed); y, p = prediksi(m)
            np.savez(f'{OUT}/runs/{tag}.npz', y=y, p=p, epoch=ep)
            if aug and seed == SEEDS[0]: torch.save(m.state_dict(), f'{OUT}/runs/{nama}.pt')
            print(f'{tag}: acc={accuracy_score(y,p.argmax(1))*100:.2f}% | epoch terbaik {ep} | {time.time()-t0:.0f} dtk')
            del m; torch.cuda.empty_cache()

## 6. Rata-rata ± SD antar-seed + interval kepercayaan 95% (bootstrap)

In [ ]:
rng = np.random.default_rng(0)
def boot_ci(y, yp, fn):
    n=len(y); s=[fn(y[i], yp[i]) for i in (rng.integers(0,n,n) for _ in range(N_BOOT))]
    return np.percentile(s,2.5), np.percentile(s,97.5)
rows=[]
for nama in MODEL:
    for aug in SKENARIO_AUG:
        for seed in SEEDS:
            d=np.load(f"{OUT}/runs/{nama}_{'aug' if aug else 'noaug'}_s{seed}.npz"); y, yp = d['y'], d['p'].argmax(1)
            pr, rc, f1, _ = precision_recall_fscore_support(y, yp, average='macro', zero_division=0)
            lo, hi = boot_ci(y, yp, accuracy_score)
            rows.append(dict(Model=nama, Augmentasi='Ya' if aug else 'Tidak', Seed=seed, Accuracy=accuracy_score(y,yp)*100,
                             Precision=pr*100, Recall=rc*100, F1=f1*100, Acc_CI_low=lo*100, Acc_CI_high=hi*100, Epoch=int(d['epoch'])))
per_run = pd.DataFrame(rows).round(2); per_run.to_csv(f'{OUT}/tabel_per_run.csv', index=False)
ring = per_run.groupby(['Model','Augmentasi'])[['Accuracy','Precision','Recall','F1']].agg(['mean','std']).round(2)
ring.columns = [f'{a} {b}' for a,b in ring.columns]; ring = ring.reset_index()
ring.to_csv(f'{OUT}/tabel_ringkasan_seed.csv', index=False); display(ring); display(per_run)

## 7. Uji statistik berpasangan (McNemar, koreksi Holm) pada citra uji yang sama

In [ ]:
pairs = [('MobileNetV3-Small','EfficientNet-B0'), ('MobileNetV3-Small','ShuffleNetV2'), ('EfficientNet-B0','ShuffleNetV2')]
rows=[]
for seed in SEEDS:
    for a,b in pairs:
        da=np.load(f'{OUT}/runs/{a}_aug_s{seed}.npz'); db=np.load(f'{OUT}/runs/{b}_aug_s{seed}.npz')
        ca = da['p'].argmax(1)==da['y']; cb = db['p'].argmax(1)==db['y']
        tbl = [[np.sum(ca&cb), np.sum(ca&~cb)], [np.sum(~ca&cb), np.sum(~ca&~cb)]]
        r = mcnemar(tbl, exact=True)
        rows.append(dict(Seed=seed, Model_A=a, Model_B=b, A_benar_B_salah=tbl[0][1], A_salah_B_benar=tbl[1][0], p_value=r.pvalue))
mc = pd.DataFrame(rows); mc['p_Holm'] = multipletests(mc['p_value'], method='holm')[1]
mc['Signifikan (0.05)'] = mc['p_Holm'] < 0.05; mc = mc.round(4)
mc.to_csv(f'{OUT}/tabel_mcnemar.csv', index=False); display(mc)
# augmentasi vs tanpa augmentasi, per model (R3-3)
rows=[]
for nama in MODEL:
    for seed in SEEDS:
        da=np.load(f'{OUT}/runs/{nama}_aug_s{seed}.npz'); db=np.load(f'{OUT}/runs/{nama}_noaug_s{seed}.npz')
        ca = da['p'].argmax(1)==da['y']; cb = db['p'].argmax(1)==db['y']
        tbl=[[np.sum(ca&cb), np.sum(ca&~cb)], [np.sum(~ca&cb), np.sum(~ca&~cb)]]
        rows.append(dict(Model=nama, Seed=seed, Selisih_acc=(ca.mean()-cb.mean())*100, p_value=mcnemar(tbl, exact=True).pvalue))
ma = pd.DataFrame(rows); ma['p_Holm'] = multipletests(ma['p_value'], method='holm')[1]; ma = ma.round(4)
ma.to_csv(f'{OUT}/tabel_augmentasi_mcnemar.csv', index=False); display(ma)

## 8. Benchmark waktu inferensi GPU & CPU (R1-1, R1-2)
Batch 1, 10 warm-up + 100 pengukuran. GPU memakai `torch.cuda.synchronize()`. Dua mode: **model saja** dan **end-to-end** (baca file + preprocessing + model).

In [ ]:
contoh = test_df['path'].tolist()[:100]
def bench(nama, dev, end2end, n=100, warm=10):
    m = buat_model(nama, len(KELAS)); m.load_state_dict(torch.load(f'{OUT}/runs/{nama}.pt', map_location='cpu'))
    m = m.to(dev).eval(); x0 = tf_plain(Image.open(contoh[0]).convert('RGB')).unsqueeze(0).to(dev); t=[]
    with torch.no_grad():
        for i in range(warm + n):
            if dev=='cuda': torch.cuda.synchronize()
            s = time.perf_counter()
            x = tf_plain(Image.open(contoh[i % len(contoh)]).convert('RGB')).unsqueeze(0).to(dev) if end2end else x0
            m(x)
            if dev=='cuda': torch.cuda.synchronize()
            if i >= warm: t.append((time.perf_counter()-s)*1000)
    return np.mean(t), np.std(t), np.median(t)
torch.set_num_threads(os.cpu_count())
rows=[]
for nama in MODEL:
    for dev in (['cuda'] if DEVICE=='cuda' else []) + ['cpu']:
        for e2e in [False, True]:
            mu, sd, med = bench(nama, dev, e2e)
            rows.append(dict(Model=nama, Perangkat='GPU T4' if dev=='cuda' else f'CPU ({os.cpu_count()} vCPU)',
                             Mode='end-to-end' if e2e else 'model saja', Mean_ms=mu, SD_ms=sd, Median_ms=med))
lat = pd.DataFrame(rows).round(2); lat.to_csv(f'{OUT}/tabel_latensi.csv', index=False); display(lat)
cpu_info = subprocess.run("lscpu | grep 'Model name'", shell=True, capture_output=True, text=True).stdout.strip()
print(cpu_info)

## 9. Memori puncak saat inferensi (R1-3)
GPU: `torch.cuda.max_memory_allocated`. CPU: puncak RSS proses terpisah per model (dikurangi baseline proses kosong).

In [ ]:
rows=[]
skrip = '''
import sys, resource, torch
from torchvision import models
import torch.nn as nn
nama, ckpt, n, mode = sys.argv[1], sys.argv[2], int(sys.argv[3]), sys.argv[4]
if mode == 'baseline':
    print(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss); sys.exit()
if nama=='MobileNetV3-Small':
    m=models.mobilenet_v3_small(); m.classifier[3]=nn.Linear(m.classifier[3].in_features,n)
elif nama=='EfficientNet-B0':
    m=models.efficientnet_b0(); m.classifier[1]=nn.Linear(m.classifier[1].in_features,n)
else:
    m=models.shufflenet_v2_x1_0(); m.fc=nn.Linear(m.fc.in_features,n)
m.load_state_dict(torch.load(ckpt, map_location='cpu')); m.eval()
with torch.no_grad():
    for _ in range(20): m(torch.randn(1,3,224,224))
print(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss)
'''
open('/content/mem_probe.py','w').write(skrip)
base = int(subprocess.run([sys.executable,'/content/mem_probe.py','x','x',str(len(KELAS)),'baseline'],capture_output=True,text=True).stdout.strip())
for nama in MODEL:
    out = subprocess.run([sys.executable,'/content/mem_probe.py',nama,f'{OUT}/runs/{nama}.pt',str(len(KELAS)),'run'],capture_output=True,text=True)
    cpu_mb = (int(out.stdout.strip()) - base)/1024
    gpu_mb = None
    if DEVICE=='cuda':
        m = buat_model(nama, len(KELAS)); m.load_state_dict(torch.load(f'{OUT}/runs/{nama}.pt')); m = m.cuda().eval()
        torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
        with torch.no_grad():
            for _ in range(20): m(torch.randn(1,3,224,224, device='cuda'))
        torch.cuda.synchronize(); gpu_mb = torch.cuda.max_memory_allocated()/1024**2; del m
    rows.append(dict(Model=nama, Parameter_M=sum(p.numel() for p in buat_model(nama,len(KELAS)).parameters())/1e6,
                     Ukuran_file_MB=os.path.getsize(f'{OUT}/runs/{nama}.pt')/1e6,
                     Puncak_RAM_CPU_MB=cpu_mb, Puncak_memori_GPU_MB=gpu_mb))
mem = pd.DataFrame(rows).round(2); mem.to_csv(f'{OUT}/tabel_memori.csv', index=False); display(mem)

## 10. Verifikasi manusia berbasis confidence (R2-3)
Prediksi dengan confidence di bawah ambang dikirim ke penjual untuk dicek. Dirata-rata atas 3 seed, model dengan augmentasi.

In [ ]:
ambang = [0.5, 0.6, 0.7, 0.8, 0.9, 0.95]; rows=[]
for nama in MODEL:
    for t in ambang:
        cov=[]; acc=[]
        for seed in SEEDS:
            d=np.load(f'{OUT}/runs/{nama}_aug_s{seed}.npz'); conf=d['p'].max(1); ok=d['p'].argmax(1)==d['y']; auto=conf>=t
            cov.append(auto.mean()*100); acc.append(ok[auto].mean()*100 if auto.any() else np.nan)
        rows.append(dict(Model=nama, Ambang=t, Otomatis_persen=np.mean(cov), Ke_verifikasi_manusia_persen=100-np.mean(cov),
                         Akurasi_otomatis=np.nanmean(acc)))
cf = pd.DataFrame(rows).round(2); cf.to_csv(f'{OUT}/tabel_confidence.csv', index=False); display(cf)
fig, ax = plt.subplots(figsize=(6.5,4.2))
for nama in MODEL:
    s = cf[cf.Model==nama]; ax.plot(s.Otomatis_persen, s.Akurasi_otomatis, 'o-', label=nama)
    for _, r in s.iterrows(): ax.annotate(f"{r.Ambang}", (r.Otomatis_persen, r.Akurasi_otomatis), fontsize=7, xytext=(3,3), textcoords='offset points')
ax.set_xlabel('Coverage: images categorized automatically (%)'); ax.set_ylabel('Accuracy on auto-categorized images (%)')
ax.invert_xaxis(); ax.grid(alpha=.3); ax.legend(fontsize=8); plt.tight_layout(); plt.savefig(f'{OUT}/gambar_confidence.png', dpi=300); plt.show()

## 11. Simpan kode untuk repositori (R2-5) dan unduh semua hasil

In [ ]:
info = dict(kelas=KELAS, split_seed=SPLIT_SEED, seeds=SEEDS, epoch_maks=EPOCH, batch=BATCH_SIZE, lr=LEARNING_RATE,
            optimizer='Adam', gpu=torch.cuda.get_device_name(0) if DEVICE=='cuda' else None, cpu=cpu_info,
            cpu_threads=os.cpu_count(), torch=torch.__version__, python=platform.python_version())
json.dump(info, open(f'{OUT}/pengaturan_eksperimen.json','w'), indent=2)
shutil.copy('/content/mem_probe.py', f'{OUT}/kode/mem_probe.py')
try:
    from google.colab import _message
    nb = _message.blocking_request('get_ipynb')['ipynb']; json.dump(nb, open(f'{OUT}/kode/eksperimen_revisi.ipynb','w'))
except Exception as e: print('Simpan notebook manual: File > Download .ipynb', e)
for f in os.listdir(f'{OUT}/runs'):
    if f.endswith('.pt'): os.remove(f'{OUT}/runs/{f}')
shutil.make_archive('/content/hasil_revisi_jtie', 'zip', OUT)
from google.colab import files; files.download('/content/hasil_revisi_jtie.zip')